In [ ]:
def get_params(config):

    d_model = config['hidden_size']
    num_layers = config['num_hidden_layers']
    vocab_size = config['vocab_size']
    num_heads = config['num_attention_heads']
    num_kv_heads = config['num_key_value_heads']
    ffw_size = config['intermediate_size']
    head_dim = config['head_dim']

    # Embedding layer (shared with output layer due to weight tying)
    embedding_params = vocab_size * d_model

    # Per layer calculations:
    def single_layer_params():
        # 1. Attention block
        # QKV projections (with GQA)
        # Q projection: (d_model * (head_dim * num_heads))
        # K,V projections: (d_model * (head_dim * num_kv_heads)) * 2
        qkv_params = (d_model * (head_dim * num_heads)) + \
                     (d_model * (head_dim * num_kv_heads) * 2)

        # Output projection
        out_proj = d_model * d_model

        # 2. Feed-forward block
        # Using GeGLU - requires extra projection compared to standard FFN
        ffn_params = (d_model * ffw_size * 2) + (ffw_size * d_model)

        # 3. RMSNorm params (2 per layer - pre-attention and pre-ffn)
        # RMSNorm has single learned parameter per feature
        rms_params = d_model * 2

        return qkv_params + out_proj + ffn_params + rms_params

    # Calculate params for all layers
    transformer_params = single_layer_params() * num_layers

    # Final RMSNorm
    final_rms_params = d_model

    # Total non-embedding parameters
    non_embedding_params = transformer_params + final_rms_params

    # Total parameters (note: output layer parameters are shared with input embeddings)
    total_params = embedding_params + non_embedding_params

    return total_params, embedding_params, non_embedding_params

config = {
    "hidden_size": 2304,
    "num_hidden_layers": 26,
    "vocab_size": 288256, # replace
    "num_attention_heads": 8,
    "num_key_value_heads": 4,
    "intermediate_size": 9216,
    "head_dim": 256,
    "max_position_embeddings": 2048
}

total, emb, non_emb = get_params(config)
print(f"Total parameters: {total:,}")
print(f"Embedding parameters: {emb:,}")
print(f"Non-embedding parameters: {non_emb:,}")
print(f"Total parameters (B): {total/1e9:.2f}B")

import math

def calculate_training_resources(config, gpu_size_gb=48, mixed_precision=True):
    d_model = config['hidden_size']
    num_layers = config['num_hidden_layers']
    seq_len = config['max_position_embeddings']

    model_size_b = total / 1e9

    if mixed_precision:
        # Formula: model_size_in_B * 1B * 1.25 / gpu_size_in_GB for training
        min_gpus_training = math.ceil(model_size_b * 18 * 1.25 / gpu_size_gb)
        # Formula: model_size_in_B * 2 * 1.25 / gpu_size_in_GB for inference
        min_gpus_inference = math.ceil(model_size_b * 2 * 1.25 / gpu_size_gb)
    else:
        pass # TODO: implement for bf16 / fp32

    # Define activation factor - was missing in the original code
    activation_factor = 1.5  # Approximate factor for activation checkpointing

    def calculate_memory_per_gpu(batch_size=1):
        """calculate memory per GPU including model states and activations"""
        # Model memory states considering FSDP and mixed precision
        if mixed_precision:
            # FP16 weights, FP32 master weights, Adam optimizer states (sharded by FSDP)
            model_states = (
                2 * model_size_b + # FP16 weights
                4 * model_size_b + # FP32 master weights
                8 * model_size_b * 2 # Adam states (m and v) (total for FSDP)
            )
        else:
            model_states = (
                4 * model_size_b + # FP32 weights
                8 * model_size_b * 2 # Adam states (no sharding)
            )

        # Memory for activations (considering gradient checkpointing and activation checkpointing)
        # Gradient checkpointing reduces activation memory by recomputing during backward pass
        # Activation checkpointing ~ 1.25 to 2.25x for activations (very very approximate)
        activation_memory = (
            (batch_size * seq_len * d_model * num_layers) *  # Fixed: added '*' instead of using parentheses as a function call
            (2 if mixed_precision else 4) * activation_factor
        ) / (1024 ** 3)  # Convert to GB

        return model_states + activation_memory

    def estimate_batch_size():
        """Estimate the optimal batch size per GPU based on memory limits."""
        memory_per_gpu = gpu_size_gb * 0.85  # leave 15% margin
        return max(1, int(memory_per_gpu / calculate_memory_per_gpu(1)))

    recommended_batch_size = estimate_batch_size()

    return {
        "model_size_billions": model_size_b,
        "model_size": model_size_b,
        "min_gpus_training": min_gpus_training,
        "min_gpus_inference": min_gpus_inference,
        "memory_per_gpu_gb": calculate_memory_per_gpu(recommended_batch_size),
        "recommended_batch_size": recommended_batch_size,
        "recommended_gpus": max(min_gpus_training, 8),
        "estimated_gpu_memory_training": min_gpus_training * gpu_size_gb,
        "training_regime": "Mixed Precision" if mixed_precision else "Full Precision"
    }

result = calculate_training_resources(config)
print(result)